# Прогноз прогрессирования диабета (регрессия)

### 1. Загрузка датасета
Датасет `load_diabetes` встроен в `sklearn.datasets`: 442 пациента, 10 признаков, целевая переменная — показатель прогрессирования заболевания через год после исходного уровня.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

data = load_diabetes()

### 2. Описание датасета

In [2]:
print(data.DESCR)

.. _diabetes_dataset:

Diabetes dataset
----------------

Ten baseline variables, age, sex, body mass index, average blood
pressure, and six blood serum measurements were obtained for each of n =
442 diabetes patients, as well as the response of interest, a
quantitative measure of disease progression one year after baseline.

**Data Set Characteristics:**

:Number of Instances: 442

:Number of Attributes: First 10 columns are numeric predictive values

:Target: Column 11 is a quantitative measure of disease progression one year after baseline

:Attribute Information:
    - age     age in years
    - sex
    - bmi     body mass index
    - bp      average blood pressure
    - s1      tc, total serum cholesterol
    - s2      ldl, low-density lipoproteins
    - s3      hdl, high-density lipoproteins
    - s4      tch, total cholesterol / HDL
    - s5      ltg, possibly log of serum triglycerides level
    - s6      glu, blood sugar level

Note: Each of these 10 feature variables have bee

### 3. Подготовка данных
Отделяем целевой признак и делим выборку на тренировочную и валидационную части в соотношении 80/20.

In [3]:
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(353, 10) (89, 10)


### 4. Обучение решающего дерева и сравнение с простой моделью
В качестве простой модели берём предсказание средним значением целевой переменной на обучающей части. Дополнительно обучим дерево с ограниченной глубиной.

In [4]:
def evaluate(model):
    model.fit(X_train, y_train)
    row = {}
    for name, Xs, ys in [('train', X_train, y_train), ('test', X_test, y_test)]:
        pred = model.predict(Xs)
        row[name + ' MSE'] = mean_squared_error(ys, pred)
        row[name + ' MAE'] = mean_absolute_error(ys, pred)
        row[name + ' R2'] = r2_score(ys, pred)
    return row

results = pd.DataFrame({
    'Среднее значение': evaluate(DummyRegressor(strategy='mean')),
    'Дерево (без ограничений)': evaluate(DecisionTreeRegressor(random_state=42)),
    'Дерево (max_depth=3)': evaluate(DecisionTreeRegressor(max_depth=3, random_state=42)),
}).T
results.round(3)

,train MSE,train MAE,train R2,test MSE,test MAE,test R2
Среднее значение,6076.398,66.448,0.000,5361.533,64.006,-0.012
Дерево (без ограничений),0.000,0.000,1.000,4976.798,54.528,0.061
Дерево (max_depth=3),2935.040,43.682,0.517,3552.701,48.097,0.329


### 5. Выводы

Модель, которая всегда предсказывает среднее значение по обучающей выборке, даёт на валидации MSE ≈ 5362 и R² ≈ 0. Это нижняя планка, с которой нужно сравнивать остальные модели.

Решающее дерево без ограничений идеально запоминает обучающую выборку (train MSE = 0, R² = 1), но на валидации его MSE ≈ 4977 и R² ≈ 0.06, то есть оно лишь немного лучше простой модели. Такой разрыв между train и test говорит о переобучении: дерево подстроилось под шум, а не под закономерность, поэтому адекватной эту модель назвать нельзя.

Если ограничить глубину дерева до 3, ошибка на train растёт (MSE ≈ 2935), зато на валидации падает до ≈ 3553 (R² ≈ 0.33), а разрыв между выборками заметно сокращается. Такое дерево лучше модели со средним значением примерно на треть по MSE, то есть оно действительно нашло в данных полезные зависимости. При этом R² = 0.33 означает, что объяснена лишь около трети разброса целевой переменной, так что качество остаётся скромным. Его можно улучшить подбором гиперпараметров по кросс-валидации или переходом к ансамблям.